# Kubeflow Pipelines と Agent Platform による継続的トレーニング（Continuous Training）

**学習目標:**
1. Kubeflow の事前構築済みコンポーネント（pre-built components）の使用方法を学ぶ
1. これらのコンポーネントを使用して Kubeflow パイプラインを構築する方法を学ぶ
1. Kubeflow パイプラインをコンパイル、アップロード、実行する方法を学ぶ


このラボでは、Google の事前構築済みコンポーネントを使用して、**scikit-learn** モデルのトレーニング、チューニング、デプロイを行う **Agent Platform** サービスをオーケストレーションする KFP パイプラインを構築、デプロイ、実行します。

## セットアップ

In [ ]:
from datetime import datetime

from google.cloud import aiplatform

In [ ]:
REGION = "us-central1"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]

## パイプライン設計の理解

パイプラインによって実装されるワークフローは、Python ベースのドメイン固有言語（DSL）を使用して定義されます。パイプラインの DSL は、以下で生成する `pipeline/pipeline_prebuilt.py` ファイルに記述されています。

パイプラインの DSL は、ファイルパスや接続文字列などの環境固有の設定のハードコードを避けるように設計されています。これらの設定は、一連の環境変数を介してパイプラインコードに提供されます。

### トレーナーイメージのビルド

パイプライン内のトレーニングステップには、カスタムトレーニングコンテナが必要です。カスタムトレーニングイメージは `trainer_image/Dockerfile` で定義されています。

In [ ]:
!cat trainer_image/Dockerfile

それでは、このトレーナーコンテナをビルドして Artifact Registry に push しましょう:

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image_covertype"
IMAGE_TAG = "latest"
TRAINING_CONTAINER_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"
TRAINING_CONTAINER_IMAGE_URI

In [ ]:
!gcloud builds submit --timeout 15m --tag $TRAINING_CONTAINER_IMAGE_URI trainer_image

モデルのサービング時にトレーニング時に使用した ML フレームワークのバージョンと一致させるために、以下のサービングコンテナをパイプラインに提供する必要があります:

In [ ]:
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)

**注:** トレーニング ML フレームワークのバージョンを変更する場合は、一致するバージョンのサービングコンテナを提供する必要があります（[予測用の事前構築済みコンテナ](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers) を参照）。

## パイプラインの構築とデプロイ

### 演習

In [ ]:
# 以下で pipeline_prebuilt.py を完成させてください:


In [ ]:
%%writefile ./pipeline/pipeline_prebuilt.py
# Copyright 2021 Google LLC

# Licensed under the Apache License, Version 2.0 (the "License"); you may not
# use this file except in compliance with the License. You may obtain a copy of
# the License at

# https://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS"
# BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either
# express or implied. See the License for the specific language governing
# permissions and limitations under the License.
"""Kubeflow Covertype パイプライン。"""
import os

from google.cloud.aiplatform import hyperparameter_tuning as hpt
from google_cloud_pipeline_components.types import artifact_types
from google_cloud_pipeline_components.v1.custom_job import CustomTrainingJobOp
from google_cloud_pipeline_components.v1.endpoint import (
    EndpointCreateOp,
    ModelDeployOp,
)
from google_cloud_pipeline_components.v1.hyperparameter_tuning_job import (
    HyperparameterTuningJobRunOp,
    serialize_metrics,
    serialize_parameters,
)
from google_cloud_pipeline_components.v1.model import ModelUploadOp
from kfp import dsl

from retrieve_best_hptune_component import retrieve_best_hptune_result

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
PROJECT_ID = os.getenv("PROJECT_ID")
REGION = os.getenv("REGION")

TRAINING_CONTAINER_IMAGE_URI = os.getenv("TRAINING_CONTAINER_IMAGE_URI")
SERVING_CONTAINER_IMAGE_URI = os.getenv("SERVING_CONTAINER_IMAGE_URI")
SERVING_MACHINE_TYPE = os.getenv("SERVING_MACHINE_TYPE", "e2-standard-8")

TRAINING_FILE_PATH = os.getenv("TRAINING_FILE_PATH")
VALIDATION_FILE_PATH = os.getenv("VALIDATION_FILE_PATH")

MAX_TRIAL_COUNT = int(os.getenv("MAX_TRIAL_COUNT", "5"))
PARALLEL_TRIAL_COUNT = int(os.getenv("PARALLEL_TRIAL_COUNT", "5"))
THRESHOLD = float(os.getenv("THRESHOLD", "0.6"))

PIPELINE_NAME = os.getenv("PIPELINE_NAME", "covertype")
BASE_OUTPUT_DIR = os.getenv("BASE_OUTPUT_DIR", PIPELINE_ROOT)
MODEL_DISPLAY_NAME = os.getenv("MODEL_DISPLAY_NAME", PIPELINE_NAME)


@dsl.pipeline(
    name=f"{PIPELINE_NAME}-kfp-pipeline",
    description="Kubeflow pipeline that tunes, trains, and deploys",
    pipeline_root=PIPELINE_ROOT,
)
def create_pipeline():
    worker_pool_specs = [
        {
            "machine_spec": {
                "machine_type": "e2-standard-4",
                # GPU を使用する場合は有効にします。
                # "accelerator_type": "NVIDIA_TESLA_T4",
                # "accelerator_count": 1,
            },
            "replica_count": 1,
            "container_spec": {
                "image_uri": TRAINING_CONTAINER_IMAGE_URI,
                "args": [
                    f"--training_dataset_path={TRAINING_FILE_PATH}",
                    f"--validation_dataset_path={VALIDATION_FILE_PATH}",
                    "--hptune",
                ],
            },
        }
    ]

    metric_spec = serialize_metrics({"accuracy": "maximize"})

    parameter_spec = serialize_parameters(
        {
            "alpha": hpt.DoubleParameterSpec(
                min=1.0e-4, max=1.0e-1, scale="log"
            ),
            "max_iter": hpt.DiscreteParameterSpec(
                values=[1, 2], scale="linear"
            ),
        }
    )

    hp_tuning_task = HyperparameterTuningJobRunOp(
        # TODO
    )

    best_retrieval_task = retrieve_best_hptune_result(
        project=PROJECT_ID,
        location=REGION,
        gcp_resources=hp_tuning_task.outputs["gcp_resources"],
        container_uri=TRAINING_CONTAINER_IMAGE_URI,
        training_file_path=TRAINING_FILE_PATH,
        validation_file_path=VALIDATION_FILE_PATH,
    )

    training_task = CustomTrainingJobOp( # pylint: disable=unused-variable
        # TODO

    )

    importer_spec = dsl.importer(
        artifact_uri=f"{BASE_OUTPUT_DIR}/model",
        artifact_class=artifact_types.UnmanagedContainerModel,
        metadata={"containerSpec": {"imageUri": SERVING_CONTAINER_IMAGE_URI}},
    )
    importer_spec.after(training_task)

    model_upload_task = ModelUploadOp( # pylint: disable=unused-variable
        # TODO
    )
    model_upload_task.after(training_task)

    endpoint_create_task = EndpointCreateOp( # pylint: disable=unused-variable
        # TODO

    )
    endpoint_create_task.after(model_upload_task)

    model_deploy_op = ModelDeployOp(  # pylint: disable=unused-variable
        # TODO
    )


### パイプラインのコンパイル

まず、パイプラインコンパイラに渡される環境変数を定義しましょう:

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATA_ROOT = f"{ARTIFACT_STORE}/data"

TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
BASE_OUTPUT_DIR = f"{ARTIFACT_STORE}/models/{TIMESTAMP}"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT_ID={PROJECT_ID}
%env REGION={REGION}
%env SERVING_CONTAINER_IMAGE_URI={SERVING_CONTAINER_IMAGE_URI}
%env TRAINING_CONTAINER_IMAGE_URI={TRAINING_CONTAINER_IMAGE_URI}
%env TRAINING_FILE_PATH={TRAINING_FILE_PATH}
%env VALIDATION_FILE_PATH={VALIDATION_FILE_PATH}
%env BASE_OUTPUT_DIR={BASE_OUTPUT_DIR}

`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成しましょう:

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

トレーニング用および検証用のデータセットを Google Cloud Storage からダウンロードします。
[ウォークスルーノートブック](https://github.com/GoogleCloudPlatform/asl-ml-immersion/blob/master/notebooks/kubeflow_pipelines/walkthrough/solutions/kfp_walkthrough_vertex.ipynb)で説明されている手順に従ってデータセットがすでに作成され Google Cloud Storage に保存されている場合は、このステップをスキップできます。

In [ ]:
%%bash
gcloud storage cp gs://asl-public-data/data/covertype/training/dataset.csv $TRAINING_FILE_PATH
gcloud storage cp gs://asl-public-data/data/covertype/validation/dataset.csv $VALIDATION_FILE_PATH

**注:** アーティファクトストアが事前に作成および適切に設定されていない場合、Agent Platform からのアクセスを許可するために **Cloud Shell** で次のコマンドを実行する必要がある場合があります:

```
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")
gcloud projects add-iam-policy-binding $PROJECT_ID     --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com"     --role="roles/storage.objectAdmin"
```

#### CLI コンパイラを使用してパイプラインをコンパイルする

次のコマンドを使用して、生成した Python ファイルからパイプラインを YAML 記述にコンパイルします:

In [ ]:
PIPELINE_YAML = "covertype_kfp_pipeline.yaml"

### 演習

`kfp dsl compile` コマンドラインを使用して `pipeline/pipeline.py` をコンパイルしてください:

In [ ]:
# TODO

**注:** Python SDK を使用してパイプラインをコンパイルすることもできます:

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

結果としてパイプラインファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプラインパッケージのデプロイ

### 演習

`aiplatform.PipelineJob` を使用してパイプラインを Agent Platform にアップロードして実行してください:

In [ ]:
# TODO

Copyright 2021 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.